# Declaración sobre el uso de Inteligencia Artificial Generativa

El desarrollo del código contenido en este notebook ha contado con la asistencia de herramientas de Inteligencia Artificial Generativa como apoyo en tareas de implementación, revisión, refactorización y mejora de determinadas partes del código.

No obstante, la concepción y desarrollo del proyecto han sido realizados por el autor. Esto incluye la definición de la arquitectura general de la solución, el diseño de los pipelines de procesamiento y entrenamiento, la selección y configuración de modelos y algoritmos, la preparación y estructuración de los datos, el diseño de los experimentos, la elección de métricas y criterios de evaluación, así como la definición de los procedimientos de validación y análisis de resultados.

Las propuestas de código obtenidas mediante herramientas de IA generativa no han sido incorporadas de forma automática. Todo el código utilizado en este proyecto ha sido revisado, comprendido, adaptado, integrado y probado por el autor, quien ha verificado su correcto funcionamiento y su adecuación a los objetivos y requisitos definidos para el proyecto.

Asimismo, las decisiones técnicas y metodológicas, la interpretación de los resultados, la resolución de errores, la comparación entre alternativas y la selección final de las soluciones implementadas han sido responsabilidad exclusiva del autor.

Por tanto, la Inteligencia Artificial Generativa ha sido utilizada como una herramienta de apoyo al desarrollo, sin sustituir el criterio técnico, el proceso de diseño, la validación experimental ni la toma de decisiones del autor.

# Comparación en vídeo: etiquetas manuales vs. YOLO26s

Este notebook carga el mejor checkpoint de YOLO26s entrenado a la mayor resolución disponible (1088 px) y genera un MP4 lado a lado:

- izquierda: fotogramas del conjunto de test con las etiquetas manuales;
- derecha: predicciones del modelo con su confianza.

El conjunto yolo_test contiene los fotogramas extraídos y anotados, no los vídeos completos. Por eso la comparación se reconstruye exclusivamente con los fotogramas etiquetados del vídeo elegido, a los mismos 3 FPS con los que se generó el dataset.

In [1]:
from __future__ import annotations

import re
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
from IPython.display import Video, display
from ultralytics import YOLO

## Configuración

Cambia TEST_SUBSET o VIDEO_ID para probar otro vídeo. Si VIDEO_ID es None se selecciona automáticamente el vídeo con más fotogramas anotados del subconjunto.

In [2]:
def find_root_dir() -> Path:
    """Localiza la raíz del repositorio desde Jupyter."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/datasets/yolo_test").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Inicia Jupyter dentro del proyecto."
    )


ROOT_DIR = find_root_dir()
MODEL_PATH = (
    ROOT_DIR
    / "results/yolo/dataset_v2/1088/yolo26s/weights/best.pt"
)
TEST_ROOT = ROOT_DIR / "data/datasets/yolo_test"
SOURCE_VIDEO_DIR = ROOT_DIR / "data/source_data/processed_data"
OUTPUT_DIR = ROOT_DIR / "results/yolo/test_video_comparisons"

TEST_SUBSET = "nocturn"  # Valores válidos: "diurn" o "nocturn"
VIDEO_ID = "000158"      # Usa None para elegir el vídeo con más frames
IMAGE_SIZE = 1088           # Mayor resolución entrenada disponible
CONFIDENCE = 0.25
IOU = 0.70
FPS = 3.0                   # FPS usados en scripts/preprocessing/extract_frames.py
PANEL_WIDTH = 720           # Ancho de cada mitad del MP4
HEADER_HEIGHT = 72
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Raíz: {ROOT_DIR}")
print(f"Dispositivo de inferencia: {DEVICE}")

Raíz: /home/jgaldeano/tfm
Dispositivo de inferencia: 0


In [3]:
def frame_number(path: Path) -> int:
    """Obtiene el índice de nombres como 000158_frame000001.jpg."""
    match = re.search(r"_frame(\d+)$", path.stem)
    if match is None:
        raise ValueError(f"Nombre de frame no válido: {path.name}")
    return int(match.group(1))


if TEST_SUBSET not in {"diurn", "nocturn"}:
    raise ValueError("TEST_SUBSET debe ser 'diurn' o 'nocturn'.")

IMAGE_DIR = TEST_ROOT / TEST_SUBSET / "images"
LABEL_DIR = TEST_ROOT / TEST_SUBSET / "labels"

for required_path in (MODEL_PATH, IMAGE_DIR, LABEL_DIR):
    if not required_path.exists():
        raise FileNotFoundError(f"No existe la ruta requerida: {required_path}")

all_test_frames = sorted(IMAGE_DIR.glob("*_frame*.jpg"))
video_counts = Counter(path.stem.split("_frame")[0] for path in all_test_frames)
if not video_counts:
    raise FileNotFoundError(f"No hay fotogramas JPG en {IMAGE_DIR}")

selected_video_id = VIDEO_ID or video_counts.most_common(1)[0][0]
frame_paths = sorted(
    IMAGE_DIR.glob(f"{selected_video_id}_frame*.jpg"),
    key=frame_number,
)
if not frame_paths:
    available = ", ".join(sorted(video_counts))
    raise ValueError(
        f"El vídeo {selected_video_id} no pertenece a {TEST_SUBSET}. "
        f"Vídeos disponibles: {available}"
    )

source_video_path = SOURCE_VIDEO_DIR / f"{selected_video_id}.mp4"
missing_labels = [path.name for path in frame_paths if not (LABEL_DIR / f"{path.stem}.txt").exists()]
if missing_labels:
    raise FileNotFoundError(
        f"Faltan etiquetas para {len(missing_labels)} frames; primero: {missing_labels[0]}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MP4 = OUTPUT_DIR / f"{selected_video_id}_gt_vs_yolo26s_1088.mp4"

print(f"Vídeo test seleccionado: {selected_video_id} ({TEST_SUBSET})")
print(f"Fotogramas anotados: {len(frame_paths)}")
print(f"Duración reconstruida: {len(frame_paths) / FPS:.2f} s")
print(f"Vídeo fuente: {source_video_path}")
print(f"Salida: {OUTPUT_MP4}")

Vídeo test seleccionado: 000158 (nocturn)
Fotogramas anotados: 61
Duración reconstruida: 20.33 s
Vídeo fuente: /home/jgaldeano/tfm/data/source_data/processed_data/000158.mp4
Salida: /home/jgaldeano/tfm/results/yolo/test_video_comparisons/000158_gt_vs_yolo26s_1088.mp4


## Funciones de dibujo y codificación

In [4]:
Box = tuple[int, float, float, float, float, float | None]


def read_yolo_labels(label_path: Path, image_shape: tuple[int, ...]) -> list[Box]:
    """Convierte etiquetas YOLO normalizadas a cajas xyxy en píxeles."""
    image_height, image_width = image_shape[:2]
    boxes: list[Box] = []

    for line_number, line in enumerate(
        label_path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 5:
            raise ValueError(
                f"Etiqueta no válida en {label_path.name}, línea {line_number}: {line}"
            )

        class_id, x_center, y_center, box_width, box_height = map(float, values)
        x1 = (x_center - box_width / 2) * image_width
        y1 = (y_center - box_height / 2) * image_height
        x2 = (x_center + box_width / 2) * image_width
        y2 = (y_center + box_height / 2) * image_height
        boxes.append((int(class_id), x1, y1, x2, y2, None))

    return boxes


def prediction_boxes(result) -> list[Box]:
    """Extrae clase, caja y confianza de un resultado de Ultralytics."""
    if result.boxes is None:
        return []

    boxes: list[Box] = []
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    confidences = result.boxes.conf.detach().cpu().numpy()
    for class_id, coordinates, confidence in zip(classes, xyxy, confidences):
        x1, y1, x2, y2 = coordinates.tolist()
        boxes.append((int(class_id), x1, y1, x2, y2, float(confidence)))
    return boxes


def add_header(image: np.ndarray, title: str, subtitle: str) -> np.ndarray:
    """Añade una cabecera del mismo ancho que el panel."""
    header = np.full((HEADER_HEIGHT, image.shape[1], 3), 24, dtype=np.uint8)
    cv2.putText(
        header, title, (20, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.78,
        (245, 245, 245), 2, cv2.LINE_AA,
    )
    cv2.putText(
        header, subtitle, (20, 57), cv2.FONT_HERSHEY_SIMPLEX, 0.48,
        (180, 180, 180), 1, cv2.LINE_AA,
    )
    return np.vstack((header, image))


def render_panel(
    frame: np.ndarray,
    boxes: list[Box],
    class_names: dict[int, str],
    color: tuple[int, int, int],
    title: str,
    subtitle: str,
) -> np.ndarray:
    """Redimensiona un frame, escala sus cajas y compone un panel."""
    source_height, source_width = frame.shape[:2]
    panel_height = round(source_height * PANEL_WIDTH / source_width)
    panel = cv2.resize(frame, (PANEL_WIDTH, panel_height), interpolation=cv2.INTER_AREA)
    scale_x = PANEL_WIDTH / source_width
    scale_y = panel_height / source_height

    for class_id, x1, y1, x2, y2, confidence in boxes:
        point_1 = (round(x1 * scale_x), round(y1 * scale_y))
        point_2 = (round(x2 * scale_x), round(y2 * scale_y))
        cv2.rectangle(panel, point_1, point_2, color, 3, cv2.LINE_AA)

        class_name = class_names.get(class_id, str(class_id))
        label = class_name if confidence is None else f"{class_name} {confidence:.2f}"
        (text_width, text_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 1
        )
        text_x = max(0, min(point_1[0], panel.shape[1] - text_width - 8))
        text_y = max(text_height + 8, point_1[1])
        cv2.rectangle(
            panel,
            (text_x, text_y - text_height - 8),
            (text_x + text_width + 8, text_y + baseline),
            color,
            -1,
        )
        cv2.putText(
            panel, label, (text_x + 4, text_y - 4),
            cv2.FONT_HERSHEY_SIMPLEX, 0.52, (15, 15, 15), 1, cv2.LINE_AA,
        )

    return add_header(panel, title, subtitle)


def transcode_for_browser(raw_path: Path, output_path: Path, fps: float) -> None:
    """Convierte a H.264 para que el MP4 se reproduzca bien en el navegador."""
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path is None:
        raw_path.replace(output_path)
        print("Aviso: ffmpeg no está disponible; se conserva el códec mp4v.")
        return

    subprocess.run(
        [
            ffmpeg_path, "-y", "-loglevel", "error",
            "-i", str(raw_path),
            "-an", "-c:v", "libx264", "-preset", "medium",
            "-crf", "20", "-pix_fmt", "yuv420p",
            "-r", str(fps), "-movflags", "+faststart",
            str(output_path),
        ],
        check=True,
    )
    raw_path.unlink()

## Generar el MP4

La inferencia usa imgsz=1088. Las cajas verdes son las etiquetas manuales y las naranjas son las predicciones.

In [5]:
model = YOLO(str(MODEL_PATH))
class_names = {int(class_id): name for class_id, name in model.names.items()}

first_frame = cv2.imread(str(frame_paths[0]))
if first_frame is None:
    raise RuntimeError(f"No se pudo leer {frame_paths[0]}")
panel_height = round(first_frame.shape[0] * PANEL_WIDTH / first_frame.shape[1])
output_size = (PANEL_WIDTH * 2, panel_height + HEADER_HEIGHT)
raw_output_path = OUTPUT_MP4.with_name(f"{OUTPUT_MP4.stem}_opencv.mp4")

writer = cv2.VideoWriter(
    str(raw_output_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    output_size,
)
if not writer.isOpened():
    raise RuntimeError(f"No se pudo abrir el codificador para {raw_output_path}")

try:
    for position, image_path in enumerate(frame_paths, start=1):
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise RuntimeError(f"No se pudo leer {image_path}")
        if frame.shape[:2] != first_frame.shape[:2]:
            raise ValueError(f"Resolución inconsistente en {image_path.name}")

        label_path = LABEL_DIR / f"{image_path.stem}.txt"
        ground_truth = read_yolo_labels(label_path, frame.shape)
        result = model.predict(
            source=frame,
            imgsz=IMAGE_SIZE,
            conf=CONFIDENCE,
            iou=IOU,
            device=DEVICE,
            verbose=False,
        )[0]

        index = frame_number(image_path)
        left_panel = render_panel(
            frame, ground_truth, class_names, (80, 220, 80),
            "Etiquetas manuales (ground truth)",
            f"Vídeo {selected_video_id} · frame test {index:06d}",
        )
        right_panel = render_panel(
            frame, prediction_boxes(result), class_names, (0, 165, 255),
            "Predicción YOLO26s",
            f"imgsz={IMAGE_SIZE} · conf>={CONFIDENCE:.2f}",
        )
        comparison = np.hstack((left_panel, right_panel))
        writer.write(comparison)

        print(
            f"Procesando {position:03d}/{len(frame_paths):03d}: {image_path.name}",
            end="\r",
        )
finally:
    writer.release()

print()
transcode_for_browser(raw_output_path, OUTPUT_MP4, FPS)
print(f"MP4 generado: {OUTPUT_MP4}")
print(f"Tamaño: {OUTPUT_MP4.stat().st_size / (1024 ** 2):.1f} MB")

Procesando 061/061: 000158_frame000061.jpg
MP4 generado: /home/jgaldeano/tfm/results/yolo/test_video_comparisons/000158_gt_vs_yolo26s_1088.mp4
Tamaño: 4.5 MB


## Resultado

In [6]:
display(Video(filename=str(OUTPUT_MP4), embed=True, html_attributes="controls loop"))